# 📖 English Standard Version (ESV) Bible Companion

## User Story

I have been appointed as the Sunday School Teacher for my church and the ESV Bible as the prefered Bible version to use for teaching. I need a Bible companion in contemporary English that balances word-for-word accuracy with modern,readable English like the ESV Bible does. I want to use the knowledge I have aquired to build a new reliable Bible companion RAG AI app that is unique and instructs in modern readable English as the ESV Bible which could also be Handy for any Sunday School teacher, Bible scholar, or personal studies.

## Use cases

- Given a Sunday school topic, the companion generates bible verses that are relevant to the topic.
- Given an excerpt from the Bible, the companion generates bible verses that are relevant to the excerpt
- At the end of every answer,  the companion generates what The bible wants us to do as lessons from the bible verse found.

## Tools
langchain, gpt-4.1-nano . Gradio

ESV Bible Markdown can be found here https://github.com/lguenth/mdbible. We are using the Bible data  in by_books folder. The only difference is seprating them into two folders old_testament and new_testament respectively.



In [ ]:
#imports
import gradio as gr
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document
from dotenv import load_dotenv


load_dotenv(override=True)

In [35]:
#contants
MODEL = "gpt-4.1-nano"
DB_NAME = "esv_bible_db"

RETRIEVAL_K = 10


In [36]:
#embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [37]:
#vector store
vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": RETRIEVAL_K}
)


In [38]:
#openai llm
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [40]:
#system prompt
SYSTEM_PROMPT = """
You are a knowledgeable, assistant as the English Standard Version (ESV) Bible Companion.
You are chatting with a user about the Bible.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question according to matched context_markdown.
If you don't know the answer, say so.
Given a Sunday School topic list ESV Bible verses that are relevant to the topic.
Given an excerpt from the Bible, list ESV bible verses that are relevant to the excerpt
Finaly, list the lessons from the ESV Bible verses matched, on each point qoute the associated Bible verse.

For context, here are specific extracts from the ESV Bible that is be directly relevant to the user's question:

{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [41]:
#fetch_context
def fetch_context(question: str) -> list[Document]:
    """
    Retrieve relevant context documents for a question.
    """
    return retriever.invoke(question)

In [42]:
# combined previous questions
def combined_question(question: str, history: list[dict] | None = None) -> str:
    """
    Combine all the user's messages into a single string.
    """
    history = history or []
    prior = "\n".join(
        m["content"] for m in history if m.get("role") == "user"
    )
    return f"{prior}\n{question}" if prior else question

In [ ]:
#answer question
def answer_question(
    question: str,
    history: list[dict] | None = None
) -> tuple[str, list[Document]]:
    """
    Answer the given question with RAG.
    Returns:
        - Generated answer
        - Retrieved context documents
    """
  
    history = history or []

    # Combine question with history for better retrieval
    combined = combined_question(question, history)
    # Retrieve documents
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Build system prompt with context
    system_prompt = SYSTEM_PROMPT.format(context=context)

    # Construct messages
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    # Invoke LLM
    response = llm.invoke(messages)

    return response.content, docs

In [ ]:
#formats matched context in the UI
def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        print(doc.metadata)
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result


In [ ]:
# RAG chat UI function
def rag_chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)

In [47]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role": "user", "content": message}]


In [ ]:
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="English Standard Version (ESV) Bible Companion", theme=theme) as ui:
    gr.Markdown("# 📖 ESV Bible Companion\nAsk me anything about the Bible!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about the Bible...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(rag_chat, inputs=chatbot, outputs=[chatbot, context_markdown])

    ui.launch(inbrowser=False)